# Enterprise Data Architecture & Reconciliation
Completed internship project solution.

In [6]:

from pathlib import Path
import sqlite3
import pandas as pd

# Correct Windows project location
PROJECT_DIR = Path(
    r"C:\Users\acer\OneDrive\Desktop\Istudio\Enterprise_Data_Reconciliation"
)

DATA_DIR = PROJECT_DIR / "data"

DB_FILE = DATA_DIR / "enterprise_database.db"
LOG_FILE = DATA_DIR / "server_logs.txt"
RATES_FILE = DATA_DIR / "daily_exchange_rates.csv"

print("Project exists:", PROJECT_DIR.is_dir())
print("Data folder exists:", DATA_DIR.is_dir())
print("Database exists:", DB_FILE.is_file())
print("Log file exists:", LOG_FILE.is_file())
print("Rates file exists:", RATES_FILE.is_file())

if DATA_DIR.is_dir():
    print("\nFiles inside data folder:")
    for file in DATA_DIR.iterdir():
        print(file.name)


Project exists: True
Data folder exists: True
Database exists: True
Log file exists: True
Rates file exists: True

Files inside data folder:
daily_exchange_rates.csv
enterprise_database.db
exchange_rates_filled.csv
latest_active_users.csv
monthly_revenue.csv
reconciled_transactions.csv
server_logs.txt
top5_customers.csv


## Phase 1 — SQL extraction
Use a window function to keep only the latest customer record and then retain Active users.

In [7]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DATA_DIR / 'enterprise_database.db')
sql = '''
WITH ranked AS (
    SELECT user_id, name, status, updated_at,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY updated_at DESC) AS rn
    FROM users
)
SELECT user_id, name, status, updated_at
FROM ranked
WHERE rn = 1 AND status = 'Active'
'''
df_active_users = pd.read_sql(sql, conn, parse_dates=['updated_at'])
conn.close()


## Phase 2 — Regex extraction and error filtering

In [8]:
logs = pd.read_csv(
    LOG_FILE,
    header=None,
    names=['raw'],
    dtype=str
).iloc[1:]

error_mask = logs['raw'].str.contains('ERROR', na=False)

pattern = (
    r'\[(?P<Date>\d{4}-\d{2}-\d{2})\s+\d{2}:\d{2}:\d{2}\]'
    r'.*?payload=\{"u":"(?P<User_ID>[^"]+)",\s*'
    r'"item_code":"(?P<Product_ID>[^"]+)",\s*'
    r'"eur_val":(?P<Euro_Value>[-\d.]+)\}'
)

extracted_data = logs.loc[~error_mask, 'raw'].str.extract(pattern)

extracted_data['Date'] = pd.to_datetime(extracted_data['Date'])
extracted_data['Euro_Value'] = pd.to_numeric(
    extracted_data['Euro_Value']
)

print("Raw logs:", len(logs))
print("ERROR logs removed:", error_mask.sum())
print("Valid parsed transactions:", len(extracted_data))

Raw logs: 100000
ERROR logs removed: 5098
Valid parsed transactions: 94902


In [9]:
## Phase 3 — Temporal and customer reconciliation

rates = pd.read_csv(
    RATES_FILE,
    parse_dates=['Date']
).sort_values('Date')

rates['EUR_to_USD'] = rates['EUR_to_USD'].ffill()

df_final = extracted_data.merge(
    rates[['Date', 'EUR_to_USD']],
    on='Date',
    how='left'
)

df_final = df_final.merge(
    df_active_users,
    left_on='User_ID',
    right_on='user_id',
    how='inner'
)

df_final['USD_Revenue'] = (
    df_final['Euro_Value'] * df_final['EUR_to_USD']
)

df_final['Month'] = (
    df_final['Date'].dt.to_period('M').astype(str)
)

print("Phase 3 completed")
print("Final reconciled transactions:", len(df_final))
print("Total USD revenue: ${:,.2f}".format(
    df_final['USD_Revenue'].sum()
))

Phase 3 completed
Final reconciled transactions: 53899
Total USD revenue: $148,274,622.91


## Phase 4 — CFO dashboard aggregations

In [ ]:
monthly_revenue = df_final.groupby('Month', as_index=False)['USD_Revenue'].sum()
top5_customers = (df_final.groupby(['User_ID','name'], as_index=False)['USD_Revenue'].sum() .sort_values('USD_Revenue', ascending=False).head(5))
monthly_revenue.to_csv('monthly_revenue.csv', index=False)
top5_customers.to_csv('top5_customers.csv', index=False)


## Validation results
- Raw logs: 100,000
- ERROR logs removed: 5,098
- Valid parsed transactions: 94,902
- Exchange-rate gaps forward-filled: 104
- Latest Active users: 14,183
- Final reconciled transactions: 53,899
- Total USD revenue: $148,274,622.91
